# Week 2 · Activity 2: Train Your Own Image Classifier
### PUP Santa Rosa — Data Science & Data Analytics

Train an AI model to identify **any objects or documents you choose** (e.g. valid vs invalid documents, signed vs unsigned, product A vs B — your classes), then run it over a folder of images and analyze the results.

**Set up your model first (no code):**
1. (Optional) Make a **Google Form** where people upload images, collected into a **Google Drive folder**.
2. Go to **https://teachablemachine.withgoogle.com/** → **Image Project** → create your classes and add example images (upload or webcam/video).
3. Click **Train Model**, then **Export Model → Tensorflow → Download my model**. You get a **.zip** (SavedModel + labels.txt).
4. Run the cells below. You'll just **upload that .zip** — the notebook arranges everything automatically; no folders to make.

The code is **generic**: it reads whatever classes your model has from `labels.txt`. Nothing is hard-coded to 'signature'.

## Step 1: Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import os, glob, zipfile, shutil
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from datetime import datetime
print("Libraries loaded.")

## Step 2: Upload your model .zip — it's arranged automatically
Run this and pick the `.zip` you downloaded from Teachable Machine. It unzips and finds the model + labels for you.

In [ ]:
# ============= UPLOAD & AUTO-ARRANGE YOUR MODEL =============
# Upload the .zip from Teachable Machine ("Export Model" -> Tensorflow -> "Download my model").
# This unzips it and finds the model + labels automatically. No manual folders needed.
from google.colab import files

print("Choose your Teachable Machine model .zip ...")
uploaded = files.upload()

zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
if not zip_names:
    raise SystemExit("No .zip found. Re-run this cell and select your model zip.")
zip_path = zip_names[0]

MODEL_ROOT = '/content/model'
if os.path.exists(MODEL_ROOT):
    shutil.rmtree(MODEL_ROOT)
os.makedirs(MODEL_ROOT, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(MODEL_ROOT)
print(f"Unzipped '{zip_path}' into {MODEL_ROOT}")

# Locate the model + labels automatically, no matter how the zip is nested
savedmodel = glob.glob(f'{MODEL_ROOT}/**/saved_model.pb', recursive=True)
keras_files = glob.glob(f'{MODEL_ROOT}/**/*.h5', recursive=True)
label_files = glob.glob(f'{MODEL_ROOT}/**/labels.txt', recursive=True)
labels_path = label_files[0] if label_files else None

print("Found -> SavedModel:", bool(savedmodel), "| Keras .h5:", bool(keras_files), "| labels.txt:", bool(labels_path))

## Step 3: Load the model and read your classes

In [ ]:
# ============= LOAD THE MODEL (SavedModel or Keras) =============
if savedmodel:
    MODEL_DIR = os.path.dirname(savedmodel[0])
    model = tf.saved_model.load(MODEL_DIR)
    predict_fn = model.signatures['serving_default']
    MODE = 'savedmodel'
    print(f"Loaded SavedModel from {MODEL_DIR}")
elif keras_files:
    model = tf.keras.models.load_model(keras_files[0], compile=False)
    predict_fn = None
    MODE = 'keras'
    print(f"Loaded Keras model from {keras_files[0]}")
else:
    raise SystemExit("No model found in the zip (need saved_model.pb or a .h5 file).")

# Parse class names generically: Teachable Machine writes "0 ClassName" per line.
if labels_path:
    class_names = []
    with open(labels_path) as f:
        for line in f:
            t = line.strip()
            if not t:
                continue
            parts = t.split(' ', 1)
            class_names.append(parts[1] if len(parts) == 2 and parts[0].isdigit() else t)
else:
    class_names = []
    print("No labels.txt found - classes will be shown as 'Class 0', 'Class 1', ...")

print(f"Classes ({len(class_names)}):", class_names)

## Step 4: Prediction functions

In [ ]:
# ============= PREDICTION FUNCTIONS (work for any classes) =============
def _preprocess(img):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize((224, 224))
    arr = np.asarray(img).astype(np.float32)
    if MODE == 'savedmodel':
        return (arr / 255.0)[None, ...]
    else:  # Teachable Machine Keras models expect values in [-1, 1]
        return ((arr / 127.5) - 1.0)[None, ...]

def predict_image(image_path, show_visualization=False):
    try:
        img = Image.open(image_path)
        x = _preprocess(img.copy())
        if MODE == 'savedmodel':
            out = predict_fn(tf.constant(x))
            probs = out[list(out.keys())[0]].numpy()[0]
        else:
            probs = model.predict(x, verbose=0)[0]
        idx = int(np.argmax(probs))
        label = class_names[idx] if idx < len(class_names) else f'Class {idx}'
        if show_visualization:
            visualize_prediction(img, label, float(probs[idx]), probs)
        return {'filename': os.path.basename(image_path), 'prediction': label,
                'confidence': float(probs[idx]), 'probabilities': probs, 'path': image_path}
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

def visualize_prediction(image, prediction, confidence, probabilities):
    labels = class_names if len(class_names) == len(probabilities) else [f'Class {i}' for i in range(len(probabilities))]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(image); ax1.axis('off')
    ax1.set_title(f"{prediction} ({confidence:.1%})", fontsize=14, fontweight='bold')
    ax2.barh(labels, probabilities, color='steelblue', alpha=0.8)
    ax2.set_xlim(0, 1); ax2.set_xlabel('Confidence'); ax2.set_title('Class probabilities')
    plt.tight_layout(); plt.show()

print("Prediction functions ready.")

## Step 5: Point to your images (Google Drive folder, or upload)
**Option A (Drive):** set `IMAGES_DIR` to your Drive folder — every image in it gets classified.
**Option B:** leave it `None` and upload a `.zip` of images.

In [ ]:
# ============= GET THE IMAGES TO CLASSIFY =============
# TWO WAYS to give the notebook your images. Pick one:
#
#   OPTION A - Google Drive folder (recommended for a whole folder of documents):
#     set IMAGES_DIR to your Drive folder path below. The notebook mounts Drive
#     for you and classifies every image in that folder (and its subfolders).
#
#   OPTION B - Upload a .zip (or a few image files): leave IMAGES_DIR = None.
#
IMAGES_DIR = None   # e.g. '/content/drive/MyDrive/my_documents'

if IMAGES_DIR and IMAGES_DIR.startswith('/content/drive'):
    # OPTION A: connect Google Drive and use the folder as-is
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    if not os.path.isdir(IMAGES_DIR):
        raise SystemExit(f"Folder not found: {IMAGES_DIR}  (check the path / sharing)")
    print("Using Google Drive folder:", IMAGES_DIR)
else:
    # OPTION B: upload a zip or image files
    from google.colab import files
    print("Choose a .zip of the images to classify (or select several image files) ...")
    up = files.upload()
    IMAGES_DIR = '/content/images'
    if os.path.exists(IMAGES_DIR):
        shutil.rmtree(IMAGES_DIR)
    os.makedirs(IMAGES_DIR, exist_ok=True)
    zs = [n for n in up if n.lower().endswith('.zip')]
    if zs:
        with zipfile.ZipFile(zs[0]) as z:
            z.extractall(IMAGES_DIR)
        print(f"Images extracted to {IMAGES_DIR}")
    else:
        for name in up:
            with open(os.path.join(IMAGES_DIR, name), 'wb') as fout:
                fout.write(up[name])
        print(f"Saved {len(up)} uploaded file(s) to {IMAGES_DIR}")

print("Images folder:", IMAGES_DIR)

## Step 6: Classify every image

In [ ]:
# ============= CLASSIFY EVERY IMAGE IN THE FOLDER =============
def process_folder(folder_path, review_threshold=0.8, visualize_samples=3):
    exts = ['*.jpg', '*.jpeg', '*.png', '*.gif', '*.bmp', '*.webp',
            '*.JPG', '*.JPEG', '*.PNG']
    image_files = []
    for ext in exts:
        image_files += glob.glob(os.path.join(folder_path, ext))
        image_files += glob.glob(os.path.join(folder_path, '**', ext), recursive=True)
    image_files = sorted(set(image_files))
    if not image_files:
        print("No images found in", folder_path)
        return None
    print(f"Found {len(image_files)} images. Classifying ...")
    results = []
    for i, p in enumerate(image_files, 1):
        r = predict_image(p)
        if r:
            results.append(r)
        if i % 10 == 0 or i == len(image_files):
            print(f"  {i}/{len(image_files)}")

    for r in results[:visualize_samples]:
        visualize_prediction(Image.open(r['path']), r['prediction'], r['confidence'], r['probabilities'])

    counts = Counter(r['prediction'] for r in results)
    print("\nSUMMARY")
    for cls, n in counts.most_common():
        print(f"  {cls}: {n} ({n/len(results)*100:.1f}%)")
    avg = np.mean([r['confidence'] for r in results]) if results else 0
    print(f"  Average confidence: {avg:.1%}")
    low = [r for r in results if r['confidence'] < review_threshold]
    if low:
        print(f"  Needs review (<{review_threshold:.0%}): {len(low)}")
    return {'results': results, 'counts': dict(counts), 'review_threshold': review_threshold}

batch = process_folder(IMAGES_DIR)

## Step 7: Save the results to CSV

In [ ]:
# ============= SAVE RESULTS TO CSV =============
def save_results_to_csv(batch, output_path='/content/classification_results.csv'):
    if not batch or not batch['results']:
        print("No results to save")
        return None
    thr = batch['review_threshold']
    rows = [{'Filename': r['filename'],
             'Prediction': r['prediction'],
             'Confidence': f"{r['confidence']:.2%}",
             'Confidence_Numeric': r['confidence'],
             'Needs_Review': 'Yes' if r['confidence'] < thr else 'No',
             'Full_Path': r['path']} for r in batch['results']]
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"Saved {output_path} with {len(df)} rows")
    return df

df = save_results_to_csv(batch)
if df is not None:
    display(df.head())

## Step 8: Auto-sort the images into one folder per predicted class

In [ ]:
# ============= ORGANIZE FILES INTO ONE FOLDER PER CLASS =============
def organize_files(batch, output_base='/content/organized_images'):
    if not batch or not batch['results']:
        print("Nothing to organize")
        return
    thr = batch['review_threshold']
    if os.path.exists(output_base):
        shutil.rmtree(output_base)
    for r in batch['results']:
        if r['confidence'] < thr:
            sub = 'needs_review'
        else:
            sub = ''.join(c if (c.isalnum() or c in ' _-') else '_' for c in r['prediction']).strip().replace(' ', '_')
            sub = sub or 'unknown'
        dest_dir = os.path.join(output_base, sub)
        os.makedirs(dest_dir, exist_ok=True)
        try:
            shutil.copy2(r['path'], os.path.join(dest_dir, r['filename']))
        except Exception as e:
            print("copy error:", e)
    print("Organized into", output_base)
    for d in sorted(os.listdir(output_base)):
        print(f"  {d}/: {len(os.listdir(os.path.join(output_base, d)))} files")

organize_files(batch)

## Step 9: Analyze the results (EDA)
Charts and per-class summary of what the model identified.

In [ ]:
# ============= GENERIC EDA ON THE RESULTS =============
if df is None or len(df) == 0:
    print("No data for EDA")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'Classification Summary (n={len(df)})', fontsize=14, fontweight='bold')

    classes = sorted(df['Prediction'].unique())
    cmap = {c: plt.cm.tab10(i % 10) for i, c in enumerate(classes)}
    colors = [cmap[c] for c in df['Prediction']]
    axes[0].bar(range(len(df)), df['Confidence_Numeric'], color=colors, edgecolor='black', alpha=0.85)
    axes[0].axhline(0.8, color='orange', linestyle='--', label='Review 80%')
    axes[0].set_title('Confidence per item'); axes[0].set_xlabel('Item'); axes[0].set_ylabel('Confidence')
    axes[0].set_ylim(0, 1.1); axes[0].legend()

    counts = df['Prediction'].value_counts()
    axes[1].pie(counts.values, labels=[f'{k} ({v})' for k, v in counts.items()], autopct='%1.0f%%', startangle=90)
    axes[1].set_title('Predictions by class')

    avg = df.groupby('Prediction')['Confidence_Numeric'].mean().sort_values()
    axes[2].barh(avg.index, avg.values, color='teal')
    axes[2].set_xlim(0, 1); axes[2].set_title('Avg confidence by class'); axes[2].set_xlabel('Avg confidence')

    plt.tight_layout(); plt.show()

    print("Per-class summary:")
    for cls in classes:
        sub = df[df['Prediction'] == cls]['Confidence_Numeric']
        print(f"  {cls}: {len(sub)} item(s), avg confidence {sub.mean():.1%}")
    print(f"\nNeeds review (<80%): {(df['Confidence_Numeric'] < 0.8).sum()} of {len(df)}")

---
### What you learned

You trained a no-code AI classifier on **your own classes**, uploaded the model as a single zip that the notebook arranged automatically, ran it over a whole folder of images, sorted them by prediction, and analyzed the results — the full **train → classify → organize → analyze** loop, reusable for any objects or documents.

— PUP Santa Rosa · Data Science & Data Analytics